In [2]:
import pandas as pd

df = pd.read_json("../series.jsonl", lines=True)
print(df.shape)        # (row_count, col_count)
print(df.columns.tolist())

KeyboardInterrupt: 

In [4]:
import pandas as pd

df = pd.read_json("../series.jsonl", lines=True)

print("Total entries:", len(df))
print("% null description:", round(df["description"].isna().mean() * 100), "%")

print("\nType breakdown:")
print(df["type"].value_counts())

print("\nContent rating:")
print(df["content_rating"].value_counts())

print("\nTop 30 genres:")
print(df["genres"].explode().value_counts().head(30))

print("\nTop 50 tags:")
print(df["tags"].explode().value_counts().head(50))

Total entries: 558299
% null description: 34 %

Type breakdown:
type
manga     364194
novel      61373
manhwa     59437
other      45909
manhua     24923
oel         2463
Name: count, dtype: int64

Content rating:
content_rating
safe            391329
pornographic     99157
erotica          56261
suggestive       11552
Name: count, dtype: int64

Top 30 genres:
genres
Romance          144808
Drama             98058
Hentai            92428
Comedy            84878
Adult             74931
Fantasy           71668
Yaoi              48301
Slice of Life     45625
Action            44547
Supernatural      37709
Shoujo            37643
Seinen            37186
Doujinshi         35786
Josei             34841
School Life       30271
Shounen           28141
Smut              23515
Adventure         22546
Boys Love         20838
Erotica           19978
Mystery           15591
Ecchi             15469
Shounen Ai        13241
Historical        12988
Horror            11616
Psychological     11147
Mature

In [3]:
import pandas as pd

# Load
df = pd.read_json("../series.jsonl", lines=True)

df = df[df["state"] == "active"]

# Only manga/manhwa/manhua/oel 
df = df[df["type"].isin(["manga", "manhwa", "manhua", "oel", 'other'])]

# 3. Content rating filter — adjust if you want explicit content
df = df[df["content_rating"].isin(["safe", "suggestive"])]

# 4. Normalize genres + tags to lowercase lists (they're titlecase in the API,
#    lowercase in the dump — pick one convention now and stick to it)
df["genres"] = df["genres"].apply(
    lambda x: [genre.lower() for genre in x] if isinstance(x, list) else []
)
df["tags"] = df["tags"].apply(
    lambda x: [tag.lower() for tag in x] if isinstance(x, list) else []
)

# 5. Drop columns you'll never use
KEEP = [
    "id", "title", "native_title", "type", "status",
    "year", "rating", "description", "genres", "tags",
    "cover", "authors", "total_chapters", "source",
]
df = df[[c for c in KEEP if c in df.columns]].reset_index(drop=True)

df.to_pickle("manga_clean.pkl")
print(f"Saved {len(df)} entries")

print(f"Final dataset: {len(df)} entries")
print(f"% with description: {round(df['description'].notna().mean() * 100)}%")
print(f"% with genres: {round((df['genres'].str.len() > 0).mean() * 100)}%")
print(f"% with tags: {round((df['tags'].str.len() > 0).mean() * 100)}%")

Saved 177991 entries
Final dataset: 177991 entries
% with description: 66%
% with genres: 92%
% with tags: 54%


In [9]:
import pandas as pd

df = pd.read_pickle("manga_clean.pkl")

# Show one cover dict in full
print("First cover object:")
print(df['cover'].iloc[0])

print("\n\nFirst source object:")
print(df['source'].iloc[0])

First cover object:
{'raw': {'url': 'https://images.mangabaka.dev/0/f/5/2/4/2/1/4/0904/4a7d/8373/608fb67c1b53', 'size': 536283, 'width': 580, 'format': 'jpeg', 'height': 838, 'blurhash': '|sFrSzad9Yoyt4V{RPf$ocxzaLNFoyWARRoKj[fOJ$s;xYS1Rjofofo3WC9Yj[xBWCn,oxbbWBjcIVS0s*n,t6bYbIj@j]V_WUs.nloybHoMj[WDV@f6ogWCbbjbjbWWWExXofWVa#R-oLaebHofoHa_WUofR+WBoMWVj=', 'thumbhash': 'GRYOPQgHiH+GVpgXeGR4Z3iAmGQJ'}, 'x150': {'x1': 'https://cdn.mangabaka.dev/imgproxy/plain/x150@1/aHR0cHM6Ly9pbWFnZXMubWFuZ2FiYWthLmRldi8wL2YvNS8yLzQvMi8xLzQvMDkwNC80YTdkLzgzNzMvNjA4ZmI2N2MxYjUz', 'x2': 'https://cdn.mangabaka.dev/imgproxy/plain/x150@2/aHR0cHM6Ly9pbWFnZXMubWFuZ2FiYWthLmRldi8wL2YvNS8yLzQvMi8xLzQvMDkwNC80YTdkLzgzNzMvNjA4ZmI2N2MxYjUz', 'x3': 'https://cdn.mangabaka.dev/imgproxy/plain/x150@3/aHR0cHM6Ly9pbWFnZXMubWFuZ2FiYWthLmRldi8wL2YvNS8yLzQvMi8xLzQvMDkwNC80YTdkLzgzNzMvNjA4ZmI2N2MxYjUz'}, 'x250': {'x1': 'https://cdn.mangabaka.dev/imgproxy/plain/x250@1/aHR0cHM6Ly9pbWFnZXMubWFuZ2FiYWthLmRldi8wL2YvNS8yLzQvMi8xLzQvM

In [10]:
import pandas as pd

df = pd.read_pickle("manga_clean.pkl")

# Extract cover URL
def extract_cover_url(cover):
    if not isinstance(cover, dict):
        return None
    x250 = cover.get("x250", {})
    if isinstance(x250, dict):
        url = x250.get("x1") or x250.get("x2")
        if url:
            return url
    return cover.get("default") or cover.get("small")

# Extract avg rating
def extract_avg_rating(source):
    if not isinstance(source, dict):
        return None
    ratings = []
    for provider in source.values():
        if isinstance(provider, dict):
            r = provider.get("rating")
            if r is not None and float(r) > 0:
                ratings.append(float(r))
    return round(sum(ratings) / len(ratings), 2) if ratings else None

df["cover_url"] = df["cover"].apply(extract_cover_url)
df["avg_rating"] = df["source"].apply(extract_avg_rating)

# Drop the nested columns
df = df.drop(columns=["cover", "source"])

# Save
df.to_pickle("manga_clean.pkl")

print(f"Final dataset: {len(df)} entries")
print(f"Columns: {list(df.columns)}")
print(f"% with cover URL: {round(df['cover_url'].notna().mean() * 100)}%")
print(f"% with avg rating: {round(df['avg_rating'].notna().mean() * 100)}%")

Final dataset: 177991 entries
Columns: ['id', 'title', 'native_title', 'type', 'status', 'year', 'rating', 'description', 'genres', 'tags', 'authors', 'total_chapters', 'cover_url', 'avg_rating']
% with cover URL: 98%
% with avg rating: 54%
